In [1]:
import numpy as np
from google.colab import drive
import pandas as pd
import datetime
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

# Fully scaled enhanced directory representing all organizations in the list
enhanced_directory = [
    {
        "name": "Parliamentary Budget Officer (PBO)",
        "url": "https://pbo-dpb.ca",
        "feed_url": "https://www.pbo-dpb.ca/en/feed.xml"
    },
    {
        "name": "Bank of Canada",
        "url": "https://bankofcanada.ca",
        "feed_url": "https://www.bankofcanada.ca/valet/feeds/publications/en"
    },
    {
        "name": "Statistics Canada — The Daily",
        "url": "https://statcan.gc.ca",
        "feed_url": "https://www150.statcan.gc.ca/n1/dai-quo/rss-eng.xml"
    },
    {
        "name": "C.D. Howe Institute",
        "url": "https://cdhowe.org",
        "feed_url": "https://www.cdhowe.org/rss.xml"
    },
    {
        "name": "Fraser Institute",
        "url": "https://www.fraserinstitute.org/",
        "feed_url": "https://www.fraserinstitute.org/rss.xml"
    },
    {
        "name": "Macdonald-Laurier Institute (MLI)",
        "url": "https://macdonaldlaurier.ca/",
        "feed_url": "https://macdonaldlaurier.ca/feed/"
    },
    {
        "name": "Policy Options / IRPP",
        "url": "https://policyoptions.irpp.org/",
        "feed_url": "https://policyoptions.irpp.org/feed/"
    },
    {
        "name": "Canadian Centre for Policy Alternatives (CCPA)",
        "url": "https://www.policyalternatives.ca/",
        "feed_url": "https://www.policyalternatives.ca/rss.xml"
    },
    {
        "name": "Broadbent Institute",
        "url": "https://broadbentinstitute.ca",
        "feed_url": "https://www.broadbentinstitute.ca/blog.rss"
    },
    {
        "name": "Signal49 Research",
        "url": "https://signal49.ca"
    },
    {
        "name": "TD Economics",
        "url": "https://td.com"
    },
    {
        "name": "RBC Economics",
        "url": "https://rbc.com"
    },
    {
        "name": "BMO Economics",
        "url": "https://bmo.com"
    },
    {
        "name": "Scotiabank Economics",
        "url": "https://scotiabank.com"
    },
    {
        "name": "CIBC Capital Markets Economics",
        "url": "https://cibccm.com"
    },
    {
        "name": "Desjardins Economic Studies",
        "url": "https://desjardins.com"
    },
    {
        "name": "Jack Mintz — Financial Post",
        "url": "https://financialpost.com"
    },
    {
        "name": "School of Public Policy, University of Calgary",
        "url": "https://policyschool.ca",
        "feed_url": "https://www.policyschool.ca/feed/"
    },
    {
        "name": "Canada West Foundation",
        "url": "https://cwf.ca",
        "feed_url": "https://cwf.ca/feed/"
    },
    {
        "name": "Parkland Institute",
        "url": "https://parklandinstitute.ca",
        "feed_url": "https://www.parklandinstitute.ca/blog.rss"
    },
    {
        "name": "Frontier Centre for Public Policy",
        "url": "https://fcpp.org",
        "feed_url": "https://fcpp.org/feed/"
    },
    {
        "name": "Atlantic Provinces Economic Council (APEC)",
        "url": "https://apec-econ.ca"
    },
    {
        "name": "Atlantic Institute for Market Studies (AIMS)",
        "url": "https://www.fraserinstitute.org/",
        "feed_url": "https://www.fraserinstitute.org/rss.xml"
    },
    {
        "name": "Financial Accountability Office of Ontario (FAO)",
        "url": "https://fao-on.org",
        "feed_url": "https://fao-on.org/en/blog/rss"
    },
    {
        "name": "Munk School of Global Affairs and Public Policy, University of Toronto",
        "url": "https://utoronto.ca"
    },
    {
        "name": "Institut du Québec",
        "url": "https://institutduquebec.ca"
    },
    {
        "name": "Public Policy Forum (PPF)",
        "url": "https://ppforum.ca",
        "feed_url": "https://ppforum.ca/feed/"
    },
    {
        "name": "Institut économique de Montréal (IEDM / MEI)",
        "url": "https://iedm.org",
        "feed_url": "https://www.iedm.org/feed/"
    },
    {
        "name": "CFFP — Université de Sherbrooke",
        "url": "https://usherbrooke.ca"
    },
    {
        "name": "CIRANO",
        "url": "https://cirano.qc.ca"
    },
    {
        "name": "Institut de la statistique du Québec (ISQ)",
        "url": "https://quebec.ca"
    },
    {
        "name": "Conseil du patronat du Québec (CPQ)",
        "url": "https://cpq.qc.ca"
    },
    {
        "name": "International Monetary Fund (IMF)",
        "url": "https://imf.org",
        "feed_url": "https://www.imf.org/en/News/RSS"
    },
    {
        "name": "Better Dwelling",
        "url": "https://betterdwelling.com",
        "feed_url": "https://betterdwelling.com/feed/"
    },
    {
        "name": "USA Facts",
        "url": "https://usafacts.org"
    },
    {
        "name": "OECD",
        "url": "https://oecd.org"
    },
    {
        "name": "United Nations",
        "url": "https://un.org",
        "feed_url": "https://www.un.org/press/en/feed"
    }
]

def parse_rss_feed(feed_url):
    """Parses an RSS or Atom feed using BeautifulSoup with XML parsing support."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    articles = []
    try:
        response = requests.get(feed_url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'xml')
            items = soup.find_all('item')
            if not items:
                items = soup.find_all('entry')

            for item in items:
                title_tag = item.find('title')
                link_tag = item.find('link')
                date_tag = item.find('pubDate') or item.find('pubdate') or item.find('published') or item.find('updated')

                title = title_tag.get_text(strip=True) if title_tag else None
                link = None
                if link_tag:
                    link = link_tag.get_text(strip=True) or link_tag.get('href') or link_tag.get('url')

                date_str = date_tag.get_text(strip=True) if date_tag else None

                if title and link:
                    articles.append({"Title": title, "URL": link, "Raw_Date": date_str})
    except Exception as e:
        pass
    return articles

def parse_html_fallback(url):
    """Fallback layout-agnostic HTML scraper using common structural tags."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    articles = []
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        for heading in soup.find_all(['h1', 'h2', 'h3', 'h4']):
            link_tag = heading.find('a', href=True)
            if link_tag:
                title = link_tag.get_text(strip=True)
                link = urljoin(url, link_tag['href'])
                date_str = None

                parent = heading.parent
                if parent:
                    time_tag = parent.find(['time', 'span'], class_=lambda x: x and any(d in x.lower() for d in ['date', 'time', 'meta']))
                    if time_tag:
                        date_str = time_tag.get('datetime') or time_tag.get_text(strip=True)

                if len(title) > 15 and link not in [a['URL'] for a in articles]:
                    articles.append({"Title": title, "URL": link, "Raw_Date": date_str})
    except Exception as e:
        pass
    return articles

# Date filters (set to mid-October 2026 framework)
today = pd.Timestamp('2026-10-13', tz='UTC')
start_of_week = today - pd.Timedelta(days=today.dayofweek + 1)

results = []

for org in enhanced_directory:
    print(f"Processing: {org['name']}...")
    articles = []
    if "feed_url" in org:
        articles = parse_rss_feed(org['feed_url'])

    if not articles:
        articles = parse_html_fallback(org['url'])

    if articles:
        df = pd.DataFrame(articles)
        df['Parsed_Date'] = pd.to_datetime(df['Raw_Date'], errors='coerce', utc=True)
        this_week_df = df[(df['Parsed_Date'] >= start_of_week) & (df['Parsed_Date'] <= today)]

        if not this_week_df.empty:
            print(f"   ✅ Found {len(this_week_df)} updates from this week!")
            display(this_week_df[['Title', 'Parsed_Date', 'URL']])
            for _, row in this_week_df.iterrows():
                results.append({"Organization": org['name'], "Title": row['Title'], "Date": row['Parsed_Date'], "URL": row['URL']})
        else:
            print("   ❄ No matching updates found for this week.")
    else:
        print("   ⚠️ No articles could be retrieved.")

print("\n--- Scraping Run Completed ---")


Processing: Parliamentary Budget Officer (PBO)...
   ❄ No matching updates found for this week.
Processing: Bank of Canada...
   ❄ No matching updates found for this week.
Processing: Statistics Canada — The Daily...
   ⚠️ No articles could be retrieved.
Processing: C.D. Howe Institute...
   ❄ No matching updates found for this week.
Processing: Fraser Institute...
   ❄ No matching updates found for this week.
Processing: Macdonald-Laurier Institute (MLI)...
   ❄ No matching updates found for this week.
Processing: Policy Options / IRPP...
   ❄ No matching updates found for this week.
Processing: Canadian Centre for Policy Alternatives (CCPA)...
   ❄ No matching updates found for this week.
Processing: Broadbent Institute...
   ⚠️ No articles could be retrieved.
Processing: Signal49 Research...
   ⚠️ No articles could be retrieved.
Processing: TD Economics...
   ⚠️ No articles could be retrieved.
Processing: RBC Economics...
   ⚠️ No articles could be retrieved.
Processing: BMO Economi

,Title,Parsed_Date,URL
1,This Week’s Top Stories: Canadians Migrate To ...,2026-10-11 16:08:34+00:00,https://betterdwelling.com/this-weeks-top-stor...


Processing: USA Facts...
   ⚠️ No articles could be retrieved.
Processing: OECD...
   ⚠️ No articles could be retrieved.
Processing: United Nations...
   ✅ Found 6 updates from this week!


,Title,Parsed_Date,URL
4,Syria’s Political Transition Hinges on Economi...,2026-10-12 23:58:57+00:00,https://press.un.org/en/2026/sc16474.doc.htm
5,‘Stark Gaps Remain’ in Achieving Development G...,2026-10-12 23:55:53+00:00,https://press.un.org/en/2026/gaef3624.doc.htm
6,"Humans Must Remain in Control, First Committee...",2026-10-12 23:35:53+00:00,https://press.un.org/en/2026/gadis3767.doc.htm
7,Sixth Committee Evaluates Ongoing Efforts to C...,2026-10-12 23:03:29+00:00,https://press.un.org/en/2026/gal3747.doc.htm
8,Machines Must Never Decide Whether Children Li...,2026-10-12 22:29:24+00:00,https://press.un.org/en/2026/gashc4442.doc.htm
9,Daily Press Briefing by the Office of the Spok...,2026-10-12 22:25:10+00:00,https://press.un.org/en/2026/db261012.doc.htm



--- Scraping Run Completed ---
